# Pruning and Sparsity in ONNX Models — Deep Dive

This notebook provides a rigorous treatment of **neural network pruning** and **sparsity** for ONNX model optimization.

**Topics:** Magnitude pruning, structured vs unstructured pruning, N:M sparsity, Lottery Ticket Hypothesis, sparse tensor formats, and practical deployment impact.

In [ ]:
import numpy as np
import time
from scipy import sparse
from typing import Tuple, Dict, List

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=120)

---
## 1. Magnitude Pruning

**Criterion:** Remove weights whose absolute value falls below threshold $\theta$:

$$M_{ij} = \mathbb{1}[|W_{ij}| \geq \theta], \quad W' = W \odot M$$

**Sparsity ratio:** $\rho = \frac{|\{w : w = 0\}|}{|W|} \times 100\%$

**Threshold selection:**
- **Top-k%:** $\theta = \text{Percentile}(|W|, k)$
- **Global:** single $\theta$ across all layers (small-magnitude layers pruned more)
- **Layer-wise:** per-layer $\theta_l$ for uniform sparsity

```
┌───────────────────────────────────────────────────────────┐
│           MAGNITUDE PRUNING PIPELINE                      │
│                                                           │
│  ┌──────────┐    ┌─────────────┐    ┌─────────────────┐  │
│  │  Weight   │──▶│ Sort by |w| │──▶│ Apply threshold │  │
│  │  Matrix W │    └─────────────┘    └─────────────────┘  │
│  └──────────┘              │                              │
│                            ▼                              │
│            ┌──────────────────────────────┐               │
│            │ Mask M: M_ij = 1[|W_ij|>=θ] │               │
│            └──────────────────────────────┘               │
│                            │                              │
│                            ▼                              │
│                 ┌────────────────────┐                    │
│                 │  W' = W ⊙ M        │                    │
│                 └────────────────────┘                    │
└───────────────────────────────────────────────────────────┘
```

In [ ]:
def magnitude_prune(W: np.ndarray, sparsity: float) -> Tuple[np.ndarray, np.ndarray]:
    """Apply magnitude pruning to achieve target sparsity."""
    flat = np.abs(W).flatten()
    k = int(len(flat) * sparsity)
    if k == 0:
        return W.copy(), np.ones_like(W)
    threshold = np.sort(flat)[k]
    mask = (np.abs(W) >= threshold).astype(np.float32)
    return W * mask, mask


def compute_sparsity(W: np.ndarray) -> float:
    return np.count_nonzero(W == 0) / W.size


W = np.random.randn(8, 8).astype(np.float32)
print("Original weight matrix (8x8):")
print(W)
print(f"\nOriginal sparsity: {compute_sparsity(W):.1%}")

W_pruned, mask = magnitude_prune(W, sparsity=0.5)
print(f"\nAfter 50% magnitude pruning:")
print(W_pruned)
print(f"Achieved sparsity: {compute_sparsity(W_pruned):.1%}")

In [ ]:
def global_magnitude_prune(weights: Dict[str, np.ndarray], sparsity: float) -> Dict[str, np.ndarray]:
    """Global magnitude pruning — single threshold across all layers."""
    all_mags = np.concatenate([np.abs(w).flatten() for w in weights.values()])
    threshold = np.sort(all_mags)[int(len(all_mags) * sparsity)]
    return {n: w * (np.abs(w) >= threshold).astype(np.float32) for n, w in weights.items()}


def layerwise_magnitude_prune(weights: Dict[str, np.ndarray], sparsity: float) -> Dict[str, np.ndarray]:
    """Layer-wise pruning — uniform sparsity per layer."""
    return {n: magnitude_prune(w, sparsity)[0] for n, w in weights.items()}


layers = {
    "layer1": np.random.randn(64, 128).astype(np.float32) * 0.1,
    "layer2": np.random.randn(128, 256).astype(np.float32) * 0.5,
    "layer3": np.random.randn(256, 64).astype(np.float32) * 1.0,
}

global_p = global_magnitude_prune(layers, 0.7)
layer_p = layerwise_magnitude_prune(layers, 0.7)

print("Per-layer sparsity comparison (target = 70%):")
print(f"{'Layer':<10} {'Global':>10} {'Layer-wise':>12}")
print("-" * 35)
for name in layers:
    print(f"{name:<10} {compute_sparsity(global_p[name]):>10.1%} "
          f"{compute_sparsity(layer_p[name]):>12.1%}")
print("\nGlobal pruning concentrates sparsity in small-magnitude layers.")

---
## 2. Structured vs Unstructured Pruning

```
UNSTRUCTURED — arbitrary zeros:        STRUCTURED — channel removal:
┌─────────────────────┐                ┌───────────────────────────┐
│ █ 0 █ 0 █ 0 0 █ █ 0│                │ Channel 0: █ █ █ █ █ █ █ │ ← kept
│ 0 █ 0 █ 0 █ █ 0 0 █│                │ Channel 1: ░ ░ ░ ░ ░ ░ ░ │ ← pruned
│ █ █ 0 0 █ 0 █ 0 █ 0│                │ Channel 2: ░ ░ ░ ░ ░ ░ ░ │ ← pruned
│ 0 0 █ █ 0 █ 0 █ 0 █│                │ Channel 3: █ █ █ █ █ █ █ │ ← kept
└─────────────────────┘                └───────────────────────────┘
 Same shape, scattered zeros             Reduced dims: (4,7)→(2,7)
 Needs sparse kernels for speed          Direct speedup on dense BLAS
```

| Property | Unstructured | Structured |
|----------|:------------:|:----------:|
| Granularity | Individual weights | Rows/columns/channels |
| Accuracy at same ρ | Better | Worse (coarser) |
| Speedup on dense HW | None | Direct (smaller tensors) |
| ONNX representation | Same shape, many zeros | Smaller tensor shapes |

**Group pruning criterion:** $\|W_g\|_2 = \sqrt{\sum_{(i,j) \in g} W_{ij}^2} < \theta_g \implies$ prune group $g$

In [ ]:
def structured_prune_rows(W: np.ndarray, sparsity: float) -> Tuple[np.ndarray, np.ndarray]:
    """Remove entire rows with smallest L2 norm."""
    row_norms = np.linalg.norm(W, axis=1)
    n_prune = int(len(row_norms) * sparsity)
    keep = np.sort(np.argsort(row_norms)[n_prune:])
    return W[keep], keep


def structured_prune_columns(W: np.ndarray, sparsity: float) -> Tuple[np.ndarray, np.ndarray]:
    """Remove entire columns with smallest L2 norm."""
    col_norms = np.linalg.norm(W, axis=0)
    n_prune = int(len(col_norms) * sparsity)
    keep = np.sort(np.argsort(col_norms)[n_prune:])
    return W[:, keep], keep


W = np.random.randn(8, 16).astype(np.float32)
W_unstr, _ = magnitude_prune(W, 0.5)
W_struct, kept = structured_prune_rows(W, 0.5)

print(f"Original: {W.shape}")
print(f"Unstructured: shape={W_unstr.shape}, sparsity={compute_sparsity(W_unstr):.1%}")
print(f"Structured (row): shape={W_struct.shape} — dimensions shrink!")
print(f"\nStructured pruning → fewer FLOPs on standard dense BLAS.")

---
## 3. N:M Sparsity

**N:M sparsity** enforces exactly $N$ zeros in every $M$ consecutive elements. NVIDIA Ampere+ exploits **2:4** at ~2× throughput.

```
2:4 SPARSITY PATTERN (two zeros in every four elements):

Original:  [ 0.3 | -1.2 | 0.5 | -0.1 | 0.8 | 0.2 | -0.9 | 0.4 ]
             └──── group 1 ────┘   └──── group 2 ────┘

Pruned:    [ 0.0 | -1.2 | 0.5 |  0.0 | 0.8 | 0.0 | -0.9 | 0.0 ]
             ^^^                   ^^^         ^^^           ^^^  

Hardware:  ┌──────────────────────────────────────────┐
           │ [_ █ █ _]  [█ _ _ █]  [_ █ _ █]  ...   │
           │  2 zeros    2 zeros    2 zeros          │
           │  Fixed ratio → predictable access       │
           └──────────────────────────────────────────┘
```

In [ ]:
def nm_sparsity_prune(W: np.ndarray, n: int, m: int) -> np.ndarray:
    """Apply N:M sparsity: N zeros in every M consecutive elements."""
    W_flat = W.flatten().copy()
    pad_len = (m - len(W_flat) % m) % m
    W_padded = np.concatenate([W_flat, np.zeros(pad_len)])
    groups = W_padded.reshape(-1, m)
    for i in range(len(groups)):
        groups[i, np.argsort(np.abs(groups[i]))[:n]] = 0.0
    return groups.flatten()[:len(W_flat)].reshape(W.shape)


W = np.random.randn(4, 8).astype(np.float32)
W_24 = nm_sparsity_prune(W, n=2, m=4)

print("Original:\n", W)
print(f"\nAfter 2:4 sparsity:\n", W_24)
print(f"\nSparsity: {compute_sparsity(W_24):.1%}")

groups = W_24.flatten().reshape(-1, 4)
zeros_per_group = np.sum(groups == 0, axis=1)
print(f"Zeros per group of 4: {zeros_per_group}")
print(f"Valid 2:4 pattern: {np.all(zeros_per_group == 2)}")

---
## 4. Lottery Ticket Hypothesis

> A randomly-initialized dense network $f(x; \theta_0)$ contains a sparse subnetwork $f(x; m \odot \theta_0)$ that, trained from the **same initialization**, reaches comparable accuracy.

### Iterative Magnitude Pruning (IMP)

```
┌────────────────────────────────────────────────────────────────┐
│  Round 0         Round 1         Round 2         Round k      │
│  ┌─────┐         ┌─────┐         ┌─────┐         ┌─────┐    │
│  │Init │─Train──▶│Prune│─Reset──▶│Prune│─Reset──▶│Prune│    │
│  │ θ_0 │  T ep   │ p%  │ to θ_0  │ p%  │ to θ_0  │ p%  │    │
│  └─────┘         └─────┘         └─────┘         └─────┘    │
│                     │               │               │        │
│                  Mask m₁          Mask m₂         Mask m_k   │
│                  (1-p)¹           (1-p)²          (1-p)^k    │
│                                                              │
│  Winning ticket: retrain (m_k ⊙ θ_0) from initialization    │
└────────────────────────────────────────────────────────────────┘
```

**ONNX implication:** Train sparse subnetwork → export with zeros → compress.

In [ ]:
def simulate_imp(input_dim, output_dim, prune_rate, rounds):
    """Simulate Iterative Magnitude Pruning."""
    theta_0 = np.random.randn(input_dim, output_dim).astype(np.float32) * 0.1
    mask = np.ones_like(theta_0)
    history = []
    for r in range(rounds):
        theta = (theta_0 * mask) + np.random.randn(*theta_0.shape).astype(np.float32) * 0.05
        theta *= mask
        active = np.abs(theta[mask == 1])
        if len(active) == 0:
            break
        threshold = np.sort(active)[max(1, int(len(active) * prune_rate))]
        mask[(np.abs(theta) < threshold) & (mask == 1)] = 0
        density = np.sum(mask) / mask.size
        history.append({'round': r+1, 'density': density, 'sparsity': 1-density})
    return history


history = simulate_imp(128, 64, prune_rate=0.2, rounds=10)
print("Iterative Magnitude Pruning:")
print(f"{'Round':<7} {'Density':<10} {'Sparsity':<10}")
print("-" * 28)
for h in history:
    print(f"{h['round']:<7} {h['density']:<10.3f} {h['sparsity']:<10.3f}")
print(f"\nFinal: {history[-1]['sparsity']:.1%} sparsity")

---
## 5. Theoretical Analysis

For $Y = XW$ with $X \in \mathbb{R}^{b \times d_{in}}$, $W \in \mathbb{R}^{d_{in} \times d_{out}}$:

$$\text{FLOPs}_{\text{dense}} = 2 b \cdot d_{in} \cdot d_{out}, \quad \text{FLOPs}_{\text{sparse}} = (1-\rho) \cdot \text{FLOPs}_{\text{dense}}$$

**Actual speedup** (accounting for overhead $\alpha$):
$$S_{\text{actual}} = \frac{1}{(1-\rho) + \alpha_{\text{overhead}}}$$

**Compression ratio** (CSR, float32+int32):
$$\text{CR} = \frac{\text{size}_{\text{dense}}}{\text{size}_{\text{sparse}}}, \quad \text{size}_{\text{CSR}} = (1-\rho)mn(s_v + s_i) + (m+1)s_p$$

CSR beneficial when $\rho > \frac{s_v}{s_v + s_i} = \frac{4}{8} = 50\%$

**Information-theoretic bound:**
$$\text{MDL} \geq \text{nnz} \cdot s_v + \log_2 \binom{mn}{\text{nnz}} \text{ bits}$$

In [ ]:
def analyze_compression(m, k, n, sparsity, val_bytes=4, idx_bytes=4):
    """Compute theoretical speedup and CSR compression."""
    dense_bytes = k * n * val_bytes
    nnz = int(k * n * (1 - sparsity))
    csr_bytes = nnz * (val_bytes + idx_bytes) + (k + 1) * idx_bytes
    return {
        'speedup': 1 / (1 - sparsity) if sparsity < 1 else float('inf'),
        'csr_KB': csr_bytes / 1024,
        'CR': dense_bytes / max(csr_bytes, 1),
        'csr_wins': csr_bytes < dense_bytes,
    }


print("Compression Analysis: (1024×1024) weight, batch=32")
print(f"{'Sparsity':<10} {'Speedup':>9} {'CSR(KB)':>9} {'CR':>7} {'CSR Wins?':>10}")
print("-" * 48)
for sp in [0.0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    r = analyze_compression(32, 1024, 1024, sp)
    print(f"{sp:<10.0%} {r['speedup']:>9.2f}x {r['csr_KB']:>9.1f} "
          f"{r['CR']:>7.2f}x {'Yes' if r['csr_wins'] else 'No':>10}")

---
## 6. Sparse Tensor Formats

```
Dense (4×5):              COO:                CSR:
┌─────────────────┐      rows: [0,0,1,2,2,3]  row_ptr: [0,2,3,5,6]
│ 1  0  0  2  0 │      cols: [0,3,2,1,4,0]  col_idx: [0,3,2,1,4,0]
│ 0  0  3  0  0 │      vals: [1,2,3,4,5,6]  values:  [1,2,3,4,5,6]
│ 0  4  0  0  5 │
│ 6  0  0  0  0 │      Memory: O(3·nnz)     Memory: O(2·nnz + m)
└─────────────────┘      nnz=6, sparsity=70%
```

| Format | Memory | Best For |
|--------|--------|----------|
| Dense | $O(mn)$ | Sparsity < 50% |
| COO | $O(3 \cdot \text{nnz})$ | Construction, conversion |
| CSR | $O(2 \cdot \text{nnz} + m)$ | Row-wise SpMV |
| CSC | $O(2 \cdot \text{nnz} + n)$ | Column-wise ops |
| Block Sparse | $O(\text{nnz}_{blocks} \cdot b^2)$ | GPU coalesced access |

**ONNX note:** `SparseTensorProto` uses COO. Most runtimes **densify internally**.

In [ ]:
W_demo = np.array([[1,0,0,2,0],[0,0,3,0,0],[0,4,0,0,5],[6,0,0,0,0]], dtype=np.float32)
print(f"Dense ({W_demo.shape}), {W_demo.nbytes} bytes:")
print(W_demo)

coo = sparse.coo_matrix(W_demo)
coo_b = coo.data.nbytes + coo.row.nbytes + coo.col.nbytes
print(f"\nCOO: rows={coo.row}, cols={coo.col}, vals={coo.data}")
print(f"     {coo_b} bytes (nnz={coo.nnz})")

csr = sparse.csr_matrix(W_demo)
csr_b = csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes
print(f"\nCSR: ptr={csr.indptr}, idx={csr.indices}, vals={csr.data}")
print(f"     {csr_b} bytes")

print(f"\nSparsity: {1-coo.nnz/W_demo.size:.0%} | Dense:{W_demo.nbytes}B | CSR:{csr_b}B | CR:{W_demo.nbytes/csr_b:.2f}x")

# Block sparse analysis
W_block = np.zeros((16, 16), dtype=np.float32)
W_block[0:4, 0:4] = np.random.randn(4, 4)
W_block[4:8, 8:12] = np.random.randn(4, 4)
W_block[12:16, 4:8] = np.random.randn(4, 4)
nnz_blocks = 3
block_bytes = nnz_blocks * 16 * 4 + nnz_blocks * 8
print(f"\nBlock Sparse (16×16, 4×4 blocks): {nnz_blocks}/16 blocks non-zero")
print(f"  Block sparsity: {1-nnz_blocks/16:.1%}, CR: {W_block.nbytes/block_bytes:.2f}x")

---
## 7. ONNX Model Sparsity Measurement

In [ ]:
try:
    import onnx
    from onnx import helper, TensorProto, numpy_helper
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False
    print("ONNX not installed — using numpy-only demonstration.\n")


def create_and_analyze_sparse_model(sparsity=0.8):
    """Create ONNX MLP with sparse weights and report sparsity."""
    if not HAS_ONNX:
        # Numpy fallback
        shapes = {'W1': (128,256), 'W2': (256,64), 'W3': (64,10)}
        print(f"{'Layer':<6} {'Shape':<14} {'Sparsity'}")
        print("-" * 32)
        for name, s in shapes.items():
            w, _ = magnitude_prune(np.random.randn(*s).astype(np.float32), sparsity)
            print(f"{name:<6} {str(list(s)):<14} {compute_sparsity(w):.1%}")
        return
    
    W1, _ = magnitude_prune(np.random.randn(128, 256).astype(np.float32)*0.1, sparsity)
    W2, _ = magnitude_prune(np.random.randn(256, 64).astype(np.float32)*0.1, sparsity)
    W3, _ = magnitude_prune(np.random.randn(64, 10).astype(np.float32)*0.1, sparsity*0.5)
    
    X = helper.make_tensor_value_info('input', TensorProto.FLOAT, [1, 128])
    Y = helper.make_tensor_value_info('output', TensorProto.FLOAT, [1, 10])
    graph = helper.make_graph(
        [helper.make_node('MatMul', ['input', 'W1'], ['h1']),
         helper.make_node('Relu', ['h1'], ['h1r']),
         helper.make_node('MatMul', ['h1r', 'W2'], ['h2']),
         helper.make_node('Relu', ['h2'], ['h2r']),
         helper.make_node('MatMul', ['h2r', 'W3'], ['output'])],
        'sparse_mlp', [X], [Y],
        initializer=[numpy_helper.from_array(W1, 'W1'),
                     numpy_helper.from_array(W2, 'W2'),
                     numpy_helper.from_array(W3, 'W3')])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 13)])
    onnx.checker.check_model(model)
    
    total_p, total_z = 0, 0
    print(f"{'Layer':<6} {'Shape':<14} {'Params':<8} {'Sparsity':<10} {'KB'}")
    print("-" * 46)
    for init in model.graph.initializer:
        W = numpy_helper.to_array(init)
        z = np.count_nonzero(W == 0)
        total_p += W.size; total_z += z
        print(f"{init.name:<6} {str(list(W.shape)):<14} {W.size:<8} "
              f"{z/W.size:<10.1%} {W.nbytes/1024:.1f}")
    print("-" * 46)
    print(f"{'TOTAL':<6} {'':<14} {total_p:<8} {total_z/total_p:<10.1%}")


print("ONNX Model Sparsity Report (80% target):")
create_and_analyze_sparse_model(0.8)

---
## 8. Practical Impact: Does Sparsity = Speed?

| Scenario | Speedup? | Why |
|----------|:--------:|-----|
| Unstructured + dense BLAS | **No** | All ops still computed |
| Unstructured + sparse kernel (>90%) | Sometimes | Enough to offset index overhead |
| 2:4 on Ampere GPU | Yes (~2×) | HW sparse tensor cores |
| Structured (channel pruning) | **Yes** | Tensor dimensions shrink |

```
HARDWARE SPARSE SUPPORT:
┌────────────────────┬──────────────┬──────────────┬─────────────────┐
│     Hardware       │ Unstructured │  2:4 Sparse  │   Structured    │
├────────────────────┼──────────────┼──────────────┼─────────────────┤
│ NVIDIA Ampere+     │ cuSPARSE     │ Tensor Cores │ Dense (smaller) │
│ Intel (MKL)        │ SpMV/SpMM    │ —            │ Dense (smaller) │
│ ARM (mobile)       │ Limited      │ —            │ Dense (smaller) │
│ ONNX Runtime       │ Densifies*   │ —            │ Dense (smaller) │
└────────────────────┴──────────────┴──────────────┴─────────────────┘
```

In [ ]:
def benchmark_sparsity_speedup(m, k, n, sparsity, trials=30):
    """Demonstrate that dense BLAS ignores zeros."""
    W_dense = np.random.randn(k, n).astype(np.float32)
    W_sparse, _ = magnitude_prune(np.random.randn(k, n).astype(np.float32), sparsity)
    X = np.random.randn(m, k).astype(np.float32)
    W_csr = sparse.csr_matrix(W_sparse)
    
    _ = X @ W_dense; _ = X @ W_sparse; _ = X @ W_csr
    
    t0 = time.perf_counter()
    for _ in range(trials): _ = X @ W_dense
    t_d = (time.perf_counter() - t0) / trials
    
    t0 = time.perf_counter()
    for _ in range(trials): _ = X @ W_sparse
    t_s = (time.perf_counter() - t0) / trials
    
    t0 = time.perf_counter()
    for _ in range(trials): _ = X @ W_csr
    t_k = (time.perf_counter() - t0) / trials
    
    return t_d, t_s, t_k


print("Benchmark: (32,512)@(512,512) — does sparsity help?")
print(f"{'ρ':<6} {'Dense(ms)':<10} {'Sparse+BLAS':<12} {'SparseMul':<11} {'SparseMul speedup'}")
print("-" * 55)
for sp in [0.5, 0.7, 0.9, 0.95, 0.99]:
    td, ts, tk = benchmark_sparsity_speedup(32, 512, 512, sp)
    print(f"{sp:<6.0%} {td*1000:<10.3f} {ts*1000:<12.3f} {tk*1000:<11.3f} {td/tk:.2f}x")
print("\n→ Dense BLAS: NO speedup from zeros. Sparse kernel: marginal at >95%.")

In [ ]:
def structured_speedup_demo(m, k, n, sparsity, trials=30):
    """Structured pruning gives real speedup."""
    X = np.random.randn(m, k).astype(np.float32)
    W = np.random.randn(k, n).astype(np.float32)
    W_struct = W[:, :int(n*(1-sparsity))].copy()
    _ = X @ W; _ = X @ W_struct
    
    t0 = time.perf_counter()
    for _ in range(trials): _ = X @ W
    t_orig = (time.perf_counter() - t0) / trials
    
    t0 = time.perf_counter()
    for _ in range(trials): _ = X @ W_struct
    t_struct = (time.perf_counter() - t0) / trials
    
    return t_orig, t_struct


print("Structured pruning speedup: (32,1024)@(1024,N)")
print(f"{'ρ':<6} {'Original(ms)':<13} {'Structured(ms)':<15} {'Speedup'}")
print("-" * 42)
for sp in [0.25, 0.5, 0.75, 0.9]:
    to, ts = structured_speedup_demo(32, 1024, 1024, sp)
    print(f"{sp:<6.0%} {to*1000:<13.3f} {ts*1000:<15.3f} {to/ts:.2f}x")
print("\n→ Structured pruning: REAL speedup proportional to size reduction.")

---
## 9. Sparsity Visualization & Error Analysis

In [ ]:
def visualize_ascii(W, title, max_r=12, max_c=32):
    m, n = W.shape
    rs, cs = max(1, m//max_r), max(1, n//max_c)
    print(f"\n{title} ({W.shape}, ρ={compute_sparsity(W):.0%})")
    print("┌" + "─"*min(n//cs+2, max_c+2) + "┐")
    for i in range(0, min(m, max_r*rs), rs):
        row = "│ "
        for j in range(0, min(n, max_c*cs), cs):
            row += "█" if np.any(W[i:i+rs, j:j+cs] != 0) else "·"
        print(row + " │")
    print("└" + "─"*min(n//cs+2, max_c+2) + "┘")


W_base = np.random.randn(32, 64).astype(np.float32)
visualize_ascii(magnitude_prune(W_base, 0.75)[0], "Unstructured 75%")

W_row = W_base.copy()
W_row[np.argsort(np.linalg.norm(W_row, axis=1))[:24]] = 0
visualize_ascii(W_row, "Structured Row 75%")

visualize_ascii(nm_sparsity_prune(W_base, 2, 4), "2:4 N:M Sparsity 50%")

In [ ]:
# Output error vs sparsity (without fine-tuning)
W_err = np.random.randn(256, 128).astype(np.float32)
X_err = np.random.randn(16, 256).astype(np.float32)
y_ref = X_err @ W_err

print("Output Error vs Sparsity (no fine-tuning):")
print(f"{'ρ':<6} {'Rel.Error':<10} {'Visual'}")
print("-" * 40)
for sp in np.arange(0, 1.0, 0.1):
    Wp, _ = magnitude_prune(W_err, sp)
    err = np.linalg.norm(X_err @ Wp - y_ref) / np.linalg.norm(y_ref)
    print(f"{sp:<6.0%} {err:<10.4f} {'█'*int(err*35)}")
print("\nFine-tuning after pruning recovers most accuracy.")

---
## 10. Pruning + Quantization & Deployment

$$\text{CR}_{\text{combined}} = \text{CR}_{\text{pruning}} \times \text{CR}_{\text{quantization}}$$

```
DEPLOYMENT PIPELINE:
┌────────┐    ┌──────┐    ┌──────────┐    ┌────────┐    ┌──────┐
│ Train  │──▶│ Prune │──▶│ Fine-tune │──▶│Quantize│──▶│ ONNX │
│(dense) │    │(mask) │    │(recover) │    │ (INT8) │    │export│
└────────┘    └──────┘    └──────────┘    └────────┘    └──────┘
```

In [ ]:
W_large = np.random.randn(1024, 1024).astype(np.float32)
print(f"Combined Compression (1024×1024 FP32 = {W_large.nbytes/1024:.0f} KB):")
print(f"{'ρ':<6} {'Prune CR':<10} {'INT8 CR':<9} {'Combined'}")
print("-" * 36)
for sp in [0.5, 0.7, 0.8, 0.9, 0.95]:
    Wp, _ = magnitude_prune(W_large, sp)
    nnz = np.count_nonzero(Wp)
    csr = sparse.csr_matrix(Wp)
    csr_b = csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes
    cr_p = W_large.nbytes / csr_b
    cr_q = 4.0  # FP32 → INT8
    combined_b = nnz * 1 + csr.indices.nbytes + csr.indptr.nbytes
    cr_c = W_large.nbytes / combined_b
    print(f"{sp:<6.0%} {cr_p:<10.2f}x {cr_q:<9.1f}x {cr_c:.2f}x")
print(f"\n→ Pruning(95%)+INT8 achieves massive compression for deployment.")

---
## Summary

| Concept | Formula | Impact |
|---------|---------|--------|
| Magnitude Pruning | $M_{ij} = \mathbb{1}[\|W_{ij}\| \geq \theta]$ | Simple, effective baseline |
| Group Norm | $\|W_g\|_2 = \sqrt{\sum W_{ij}^2}$ | Structured pruning criterion |
| Compression | $\text{CR} = \text{size}_{dense}/\text{size}_{sparse}$ | Storage savings |
| CSR break-even | $\rho > 50\%$ | When sparse format helps |
| Speedup bound | $S = 1/(1-\rho)$ | Upper bound (rarely achieved) |

**Key takeaways:**
1. **Unstructured sparsity ≠ speedup** on standard hardware
2. **Structured pruning** reliably reduces compute
3. **2:4 N:M** is the hardware sweet spot (NVIDIA Ampere+)
4. **ONNX Runtime** densifies sparse tensors — structured is safer
5. **Pruning + quantization** → 10-20× compression

```
DECISION FRAMEWORK:
  Want smaller file?     → Unstructured + compression
  Want faster inference?  → Structured pruning (or 2:4 on Ampere)
  Want both?             → Structured + INT8 quantization
```

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         PRUNING & SPARSITY — QUICK REFERENCE               ║
╠══════════════════════════════════════════════════════════════╣
║  TYPES:                                                      ║
║  • Unstructured: element-wise zeros, needs sparse kernels    ║
║  • Structured:   remove rows/cols/channels, shrinks dims     ║
║  • N:M:          semi-structured, hardware-friendly          ║
║                                                              ║
║  FORMULAS:                                                   ║
║  • Mask:       M_ij = 1[|W_ij| >= θ]                       ║
║  • Sparsity:   ρ = #{w=0} / |W|                            ║
║  • Group norm: ||W_g||₂ = sqrt(Σ W_ij²)                    ║
║  • CSR break-even: sparsity > 50%                            ║
║                                                              ║
║  DEPLOY:                                                     ║
║  • Speed → Structured pruning                                ║
║  • Size  → Unstructured + compression                        ║
║  • Max   → Prune + quantize (INT8)                           ║
╚══════════════════════════════════════════════════════════════╝
""")